In [66]:
from fitparse import FitFile
import pandas as pd
import numpy as np

fit_file = '22569053640_ACTIVITY.fit'

fit_file_2 = '22105359350_ACTIVITY.fit'

def get_records (fit_file,record_type,amount=None):
    all_records = [] ## Defines an array for all messages in the file
    for count,message in enumerate(fit_file.get_messages (record_type)): ## iterates through all messages of a certain type
        if amount is not None and count >= amount: ## sets a limit in case the user wants a limit
            break
        ## here I define what I want to see, a message is structured in fields. Therfore I want to have an dictionary with all fields of a message
        field_content = {
            field.name : field.value
            for field in message if field.name and 'unknown' not in field.name
        }
        all_records.append (field_content)
    return all_records

def get_dataframe (fit_file,record_type,index=None):
    if index:
        df = pd.DataFrame(get_records(fit_file,record_type)).set_index(index)
    else:
        df = pd.DataFrame(get_records(fit_file,record_type))
    return df

def find_gaps (timestamps):
    time_gap = []
    for count,timestamp in enumerate(timestamps):
        if count > 0:
            timestampdiff = timestamp - timestamps[count-1] 
            if timestampdiff.total_seconds() > 1:
                time_gap.append ({
                    'start_time': timestamps[count-1],
                    'end_time': timestamp,
                    'gap': timestampdiff.total_seconds()
                })
    return time_gap

def get_cleaned_records (fit_file,check_gaps=False):
    records = get_dataframe (fit_file,'record','timestamp')
    if check_gaps:
        print (find_gaps(records.index))
    records['cadence'] = records['cadence'].ffill(limit=10)
    records['heart_rate'] = records['heart_rate'].ffill(limit=3)
    records['power'] = records['power'].ffill(limit=1)
    return records

def get_baseline_timestamps(timestamps,baseline_minute_start,baseline_minute_end):
    if baseline_minute_start < 0 or baseline_minute_end <=baseline_minute_start:
        raise ValueError ('Invalid baseline windows negative values are not allowed')    
    ride_start = timestamps[0]

    baseline_start = ride_start + pd.Timedelta(minutes=baseline_minute_start)
    baseline_end = ride_start + pd.Timedelta(minutes=baseline_minute_end)
    return [baseline_start,baseline_end]
        
def get_baselines (parameter,dataFrame,baseline_start,baseline_end):
    baseline_timestamps = get_baseline_timestamps(sorted(dataFrame.index),baseline_start,baseline_end)
    baselines = {}
    for p in parameter:
        baselines[p] = dataFrame[p][baseline_timestamps[0]:baseline_timestamps[1]].mean()
    return baselines

def add_delta_features (dataFrame,baselines):
    df = dataFrame.copy()
    for b in baselines:
        df[f'delta_{b}'] = df[f'{b}'] - baselines[b] 
    return df

def get_slope (window):
    return np.polyfit(np.arange(len(window)),window,1)[0]

def add_slope_and_mean (dataFrame,parameters,rolling_length):
# The first 'rolling_length' rows will be NaN as the rolling function is configured to need at least
# the number of data points which is equal to the paramter length.
    df = dataFrame.copy()
    for p in parameters:
        df[f'{p}_slope'] = df[f'{p}'].rolling(rolling_length).apply(get_slope)
        df[f'{p}_mean'] = df[f'{p}'].rolling(rolling_length).mean()
    return df

def add_hr_power_delta_ratio (dataFrame,baseline):
    df = dataFrame.copy()

    hr_power_ratio_delta = (df['heart_rate']) / df['power'] - (baseline['heart_rate'] / baseline['power'])

    df ['hr_power_ratio_delta'] = hr_power_ratio_delta.replace([np.inf, -np.inf],np.nan)

    return df

def add_three_seconds_power (dataFrame):
    df = dataFrame.copy()
    df['3s_power'] = df['power'].rolling(3).mean()
    return df

def build_feature_dataframe (fit_file,baseline_start = 5, baseline_end = 15,rolling_length=300):
    activity = FitFile (fit_file)
    records = get_cleaned_records (activity)
    baselines = get_baselines (['heart_rate','power'],records,baseline_start,baseline_end)
    records = add_delta_features (records,baselines)
    records = add_slope_and_mean (records,['heart_rate','power'],300)
    records = add_hr_power_delta_ratio (records,baselines)
    records = add_three_seconds_power(records)
    return records


records = build_feature_dataframe(fit_file)





### Task 1 — Add Power CV
Power CV = rolling standard deviation divided by rolling mean, over a 15-minute window.
Your task: write a function add_power_cv(dataFrame, rolling_length) that adds a column power_cv to the DataFrame and returns it.
Acceptance criteria:

Takes a DataFrame and a window size as parameters
Adds power_cv column
Returns a new DataFrame (don't mutate the original — you already know why)
The first rolling_length - 1 rows will be NaN — that's expected and correct

One question before you write it: you already used rolling(300).mean() in week3. Given that, what do you think the formula for rolling standard deviation looks like? Take a guess, then go try it.
Come back with your code when you're done.

In [69]:
def add_power_cv (dataFrame,rolling_window = 300):
    df = dataFrame.copy()
    df ['power_cv'] = df['3s_power'].rolling(rolling_window).std() / df['3s_power'].rolling(rolling_window).mean() 
    df.loc [df['power']<20,'power_cv'] = np.nan
    return df

records = add_power_cv(records)

# print (records['power_cv'].mean())

print (records[['power','3s_power','power_cv']].iloc[300:350])

                     power    3s_power  power_cv
timestamp                                       
2026-04-18 09:33:37  130.0  140.000000       NaN
2026-04-18 09:33:38  130.0  135.333333  0.152970
2026-04-18 09:33:39  148.0  136.000000  0.152210
2026-04-18 09:33:40  150.0  142.666667  0.151610
2026-04-18 09:33:41  151.0  149.666667  0.151127
2026-04-18 09:33:42  148.0  149.666667  0.150743
2026-04-18 09:33:43  169.0  156.000000  0.150667
2026-04-18 09:33:44  176.0  164.333333  0.151127
2026-04-18 09:33:45  152.0  165.666667  0.151438
2026-04-18 09:33:46  133.0  153.666667  0.150891
2026-04-18 09:33:47  142.0  142.333333  0.149971
2026-04-18 09:33:48  144.0  139.666667  0.149008
2026-04-18 09:33:49  151.0  145.666667  0.148363
2026-04-18 09:33:50  152.0  149.000000  0.147565
2026-04-18 09:33:51  155.0  152.666667  0.146829
2026-04-18 09:33:52  160.0  155.666667  0.145960
2026-04-18 09:33:53  146.0  153.666667  0.144952
2026-04-18 09:33:54  149.0  151.666667  0.144022
2026-04-18 09:33:55 